# 03-05 Context Engineering 高级技巧

**Context Engineering** = 如何在有限的 context window 内塞入最有价值的信息。
这是 Agent 开发中最关键的工程能力，决定 Agent 的可靠性和成本。

**本节目标**：
- 理解 context window 的竞争关系
- 掌握对话历史的截断与压缩策略
- 实战 RAG 上下文注入格式
- 设计高效的 System Prompt 模板

---

In [ ]:
import os, sys
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("../.env")
from utils.llm_client import call_llm

def count_tokens(text: str) -> int:
    """简单 token 估算：中文约 1.5 chars/token，英文约 4 chars/token"""
    try:
        import tiktoken
        enc = tiktoken.encoding_for_model("gpt-4")
        return len(enc.encode(text))
    except ImportError:
        # 粗略估算（中英混合）
        chinese_chars = sum(1 for c in text if '\u4e00' <= c <= '\u9fff')
        other_chars = len(text) - chinese_chars
        return chinese_chars + other_chars // 4

print(f"Token 计数示例:")
samples = [
    "你好世界",
    "Hello World",
    "B站广告CTR优化分析报告",
    "The click-through rate optimization for Bilibili advertising",
]
for s in samples:
    print(f"  {s!r}: ~{count_tokens(s)} tokens")

## 1. Context Window 的竞争关系

```
┌─────────────────── Context Window (128K tokens) ──────────────────┐
│  System Prompt    │ History Messages │ RAG Docs │ Current │ Tools  │
│  (500-2000 tokens)│ (动态增长!)      │(0-5000)  │ Input   │ Defs   │
│                   │                  │          │(100-500)│(0-3000)│
└───────────────────────────────────────────────────────────────────┘
                              ↑
              留给模型输出的空间越来越少！
```

In [ ]:
# 量化分析 context 组成
SYSTEM_PROMPT = """你是B站商业化部门的AI广告助手。
职责：帮助广告主分析投放效果、生成广告素材、回答广告平台相关问题。
约束：
1. 不得生成违反广告法的内容（极限词、虚假宣传）
2. 所有建议需基于数据，避免主观猜测
3. 涉及竞品对比时，保持客观中立
输出格式：结构化回答，使用markdown格式，重要数据加粗。
"""

RAG_CONTEXT = """[检索到的相关文档]
文档1：B站广告CTR行业均值为2.1%，游戏类广告CTR通常高于均值，达3-5%。
文档2：影响CTR的核心因素：创意质量(40%)、用户定向精准度(30%)、投放时段(20%)、出价策略(10%)。
文档3：提升CTR的有效手段：A/B测试创意、优化人群包、调整出价。
"""

USER_QUERY = "我的游戏广告CTR只有1.2%，怎么提升？"

components = {
    "System Prompt": SYSTEM_PROMPT,
    "RAG Context":   RAG_CONTEXT,
    "User Query":    USER_QUERY,
}

total = 0
print(f"Context 组成分析:")
for name, text in components.items():
    tokens = count_tokens(text)
    total += tokens
    print(f"  {name:<15}: {tokens:>5} tokens")
print(f"  {'总计':<15}: {total:>5} tokens")
print(f"  {'剩余(128K)':<15}: {128000 - total:>5} tokens")

## 2. 对话历史管理

In [ ]:
from collections import deque
from typing import List, Dict

class ConversationHistory:
    """
    对话历史管理器 - 自动截断超出 token 预算的历史消息
    策略：保留 system + 最近 N 条消息
    """
    
    def __init__(self, max_history_tokens: int = 4000):
        self.max_tokens = max_history_tokens
        self.system: str = ""
        self.messages: deque = deque()  # (role, content) 队列
    
    def set_system(self, system: str):
        self.system = system
    
    def add(self, role: str, content: str):
        self.messages.append({"role": role, "content": content})
        self._truncate_if_needed()
    
    def _truncate_if_needed(self):
        while self._total_tokens() > self.max_tokens and len(self.messages) > 2:
            self.messages.popleft()  # 移除最旧的消息
    
    def _total_tokens(self) -> int:
        text = " ".join(m["content"] for m in self.messages)
        return count_tokens(text)
    
    def get_messages(self) -> List[Dict]:
        msgs = []
        if self.system:
            msgs.append({"role": "system", "content": self.system})
        msgs.extend(self.messages)
        return msgs
    
    def stats(self):
        return {
            "messages": len(self.messages),
            "history_tokens": self._total_tokens(),
            "system_tokens": count_tokens(self.system),
        }

# 演示：模拟 10 轮对话
hist = ConversationHistory(max_history_tokens=500)
hist.set_system("你是广告助手")

conversation = [
    ("user",      "什么是CTR？"),
    ("assistant", "CTR是点击率，= 点击数/展示数 × 100%，是衡量广告创意吸引力的核心指标。"),
    ("user",      "B站的CTR均值是多少？"),
    ("assistant", "B站广告整体CTR均值约2.1%，游戏类可达3-5%，美妆类约1.5-2.5%。"),
    ("user",      "如何提升CTR？"),
    ("assistant", "提升CTR的核心：1.优化创意素材(最重要) 2.精准定向目标用户 3.选择最佳投放时段。"),
    ("user",      "A/B测试怎么做？"),
    ("assistant", "A/B测试：同一广告主，相同预算分配给两个不同创意，跑7天后选效果好的全量投放。"),
    ("user",      "创意素材有什么要求？"),
    ("assistant", "B站广告素材：视频建议15-30秒，封面图要有强视觉冲击，文案不超过20字，避免极限词。"),
]

print(f"{'轮次':<6} {'消息数':<8} {'历史tokens':<12} {'状态':<10}")
print("-" * 38)
for i, (role, content) in enumerate(conversation):
    hist.add(role, content)
    s = hist.stats()
    status = "✅ 正常" if s['history_tokens'] <= 500 else "⚠️ 截断"
    if i % 2 == 1:  # 只打印助手回复后的状态
        print(f"{i//2+1:<6} {s['messages']:<8} {s['history_tokens']:<12} {status}")

print(f"\n最终保留 {len(hist.messages)} 条消息 (超出预算的旧消息已被丢弃)")

## 3. Context 压缩（摘要策略）

In [ ]:
def summarize_history(messages: list, max_chars: int = 200) -> str:
    """
    用 LLM 将历史对话压缩成摘要
    当 context 超出预算时，用摘要替换旧消息
    """
    history_text = "\n".join(
        f"{m['role']}: {m['content']}" for m in messages
    )
    
    prompt = f"""请将以下对话历史压缩成一段不超过{max_chars}字的摘要，
保留关键信息（用户需求、已达成的共识、重要数据）：

{history_text}

摘要："""
    
    try:
        summary = call_llm(prompt, max_tokens=300)
        return summary
    except Exception:
        # Mock 摘要
        return "[历史摘要] 用户询问了CTR相关知识：CTR均值2.1%，提升方法包括优化创意、精准定向、A/B测试。用户主要关注游戏广告优化。"

# 演示：超出预算时压缩
sample_history = [
    {"role": "user", "content": "什么是CTR？"},
    {"role": "assistant", "content": "CTR是点击率，= 点击数/展示数 × 100%"},
    {"role": "user", "content": "B站CTR均值多少？"},
    {"role": "assistant", "content": "B站整体CTR均值约2.1%，游戏类3-5%"},
]

original_tokens = count_tokens(" ".join(m["content"] for m in sample_history))
summary = summarize_history(sample_history)
compressed_tokens = count_tokens(summary)

print(f"原始历史: {original_tokens} tokens")
print(f"压缩摘要: {compressed_tokens} tokens")
print(f"压缩率: {compressed_tokens/original_tokens:.0%}")
print(f"\n摘要内容: {summary}")

## 4. RAG Context 注入格式

In [ ]:
def format_rag_context(docs: list[dict], query: str) -> str:
    """
    将检索到的文档格式化为 LLM 友好的 context
    
    Args:
        docs: [{"content": str, "source": str, "score": float}]
        query: 用户问题
    """
    context_parts = []
    context_parts.append(f"以下是与问题「{query}」相关的参考资料：")
    context_parts.append("")
    
    for i, doc in enumerate(docs, 1):
        context_parts.append(f"[参考{i}] 来源：{doc.get('source', '未知')}")
        context_parts.append(doc["content"])
        context_parts.append("")
    
    context_parts.append("请基于以上参考资料回答问题，如参考资料不足以回答，请明确说明。")
    return "\n".join(context_parts)


def build_rag_prompt(system: str, rag_context: str, query: str) -> list[dict]:
    """组装完整的 RAG prompt"""
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": f"{rag_context}\n\n问题：{query}"},
    ]


# 演示
retrieved_docs = [
    {"content": "游戏广告CTR均值3-5%，高于B站整体均值2.1%。主要受益于游戏用户高度垂直。",
     "source": "B站广告白皮书2024", "score": 0.92},
    {"content": "提升CTR的A/B测试方法：同预算、不同创意，跑7天后选胜出组。",
     "source": "广告优化实战手册", "score": 0.87},
    {"content": "游戏类广告最佳投放时段：晚上20-23点，周末全天。此时用户活跃度最高。",
     "source": "投放时段分析报告", "score": 0.81},
]

query = "游戏广告CTR只有1.2%，如何提升？"
rag_context = format_rag_context(retrieved_docs, query)

print("=== 格式化后的 RAG Context ===")
print(rag_context)
print(f"\nRAG Context token 数: {count_tokens(rag_context)}")

## 5. 高效 System Prompt 模板

好的 System Prompt 需要：角色定义 + 能力边界 + 输出格式 + 约束条件

In [ ]:
# B站广告助手的 System Prompt 模板
AD_AGENT_SYSTEM_PROMPT = """# 角色
你是B站商业化部门的AI广告优化助手，专注于帮助广告主提升投放效果。

# 核心能力
- 分析广告投放数据（CTR、CVR、ROAS、消耗等）
- 生成符合平台规范的广告文案（15字标题 + 50字描述）
- 诊断广告效果问题并给出优化建议
- 解答广告平台使用问题

# 约束
1. 广告内容禁止：极限词（最、第一、绝对）、虚假数据、竞品诋毁
2. 建议需基于数据，不做无依据的猜测
3. 不确定的问题，明确告知并建议联系平台客服

# 输出格式
- 分析类：Markdown 格式，数据加粗，结论优先
- 文案类：直接输出文本，不加多余说明
- 建议类：编号列表，按优先级排序
"""

tokens = count_tokens(AD_AGENT_SYSTEM_PROMPT)
print(f"System Prompt: {tokens} tokens")
print(f"占 GPT-4o 上下文预算: {tokens/128000*100:.1f}%")
print()

# 对比：冗长的 System Prompt vs 精简版
verbose_system = "你是一个人工智能助手，专门为B站商业化部门提供服务。你应该帮助广告主优化他们的广告投放策略。你需要分析各种广告数据指标，包括但不限于CTR点击率、CVR转化率、ROAS广告支出回报率等。你应该能够生成广告文案，诊断投放问题，提供优化建议。同时你需要遵守广告法规定，不能使用极限词语，不能进行虚假宣传。如果遇到不确定的问题，你应该明确告知用户你不确定并建议他们联系平台的客服人员获取更准确的答案。你的回答应该使用Markdown格式进行组织，重要的数字和数据应该加粗显示，建议列表应该按照优先级从高到低进行排列。"

compact_system = "你是B站广告优化AI助手。能力：数据分析/文案生成/效果诊断。约束：禁极限词，数据为据，不确定请联系客服。格式：Markdown，数据加粗，建议按优先级排列。"

print(f"冗长版 System Prompt: {count_tokens(verbose_system)} tokens")
print(f"精简版 System Prompt: {count_tokens(compact_system)} tokens")
print(f"节省: {count_tokens(verbose_system) - count_tokens(compact_system)} tokens")
print("\n精简版每次调用节省 token，长期使用可大幅降低成本！")

## Context Engineering 最佳实践

| 场景 | 策略 | 说明 |
|------|------|------|
| System Prompt | 精简 + 结构化 | 用 markdown 分节，每条约束一行，避免重复 |
| 历史消息 | 滑动窗口 + 摘要 | 超过预算时摘要旧消息，保留最近 N 轮 |
| RAG 文档 | Top-K + 相关性过滤 | 只注入相似度 > 0.7 的文档，最多 3-5 条 |
| Tool 结果 | 只保留关键字段 | 工具返回100字段，只传给LLM需要的5个 |
| 避免重复 | 引用而非复制 | "如上文所述" 而不是重复粘贴内容 |

## 面试速记

| 问题 | 要点 |
|------|------|
| Context 溢出怎么处理 | 滑动窗口截断历史 + LLM摘要旧对话 |
| 如何在有限context内塞更多信息 | 压缩System Prompt + 只注入高相关RAG + 工具结果精简 |
| KV Cache 与 context 的关系 | System Prompt 不变 → KV Cache 命中率高 → 成本降低 |
| 多轮对话怎么保持上下文 | 维护 message 列表，加入 ConversationHistory 管理器 |

**Phase 1 完成！** 下一步：`../04-rag-pipeline/01_embedding_vector_db.ipynb`